# Live TSX Portfolio Market Data Pipeline

Guided walkthrough of the latest completed run.

- Latest daily session: **2026-10-30**
- Daily rows: **28,754**
- Intraday rows: **1,426**
- Corporate actions: **328**
- Outliers flagged: **720**

The feed is a production-style near-live polling feed from Yahoo Finance, not a direct exchange tick feed.

## 1. Load the run evidence

The notebook starts from saved outputs. It does not need to download the full history again.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if (ROOT / 'data' / 'outputs').exists():
    BASE = ROOT
else:
    BASE = ROOT.parent

OUT = BASE / 'data' / 'outputs'
summary = json.loads((OUT / 'run_summary.json').read_text())
quality = pd.read_csv(OUT / 'data_quality_summary.csv')
snapshot = pd.read_csv(OUT / 'latest_risk_snapshot.csv')
weights = pd.read_csv(OUT / 'strategy_weights_latest.csv')
metrics = pd.read_csv(OUT / 'strategy_metrics_latest_window.csv')
summary

## 2. Check data quality

Coverage is measured against benchmark sessions. A warning status means review flags remain, usually outliers. It does not by itself mean ingestion failed.

In [ ]:
quality.sort_values(['coverage_pct', 'symbol'])

## 3. Read the latest risk snapshot

Volatility and drawdown use adjusted daily prices. Beta and correlation use the latest 252 daily sessions.

In [ ]:
snapshot.sort_values('realized_vol_60d', ascending=False)

## 4. Compare refreshed weights

Weights sum to 100 percent for each method, with a 20 percent position cap and a 40 percent sector cap.

In [ ]:
weights.pivot(index='short', columns='strategy', values='weight').round(4)

## 5. Read the window diagnostics carefully

These results are calculated on the same 104-week window used to select the weights. They check the optimisation. They are not forecasts and they do not revise the earlier V2 backtest.

In [ ]:
metrics

## 6. Run a new pipeline update

Run these commands from the project root:

```bash
python run_pipeline.py
python scripts/build_deliverables.py
```

A failed daily symbol stops the run before strategy outputs are refreshed.